# Nucleic-Acid Heavy-Atom RMSD

這份 Notebook 一次分析 1 個 topology + 1 個 DCD。

分析流程：

1. 選取核酸重原子，排除配體、水、離子與氫原子。
2. 將所有 frame 以相同原子對齊至第 0 frame。
3. 計算各 frame 相對於第 0 frame 的 RMSD。
4. 輸出 NPZ、CSV、PNG、PDF 與執行摘要。

MDTraj 座標單位為 nm；輸出數據同時保存 nm 與 Å。


## 1. 環境準備

需要 Python 3、MDTraj、NumPy 與 Matplotlib。

    pip install mdtraj numpy matplotlib


In [ ]:
# ============================================================
# 2. 載入套件
# ============================================================

from pathlib import Path
import csv

import mdtraj as md
import matplotlib.pyplot as plt
import numpy as np

print(f"MDTraj: {md.__version__}")
print(f"NumPy: {np.__version__}")


## 3. 使用者設定區

一般情況只需修改 TOPOLOGY_PATH、DCD_PATH、SYSTEM_NAME 與 OUTPUT_DIR。

本研究的 DCD 每個原始 frame 代表 0.002 ns。若 TRAJECTORY_STRIDE = 10，繪圖時間間隔會自動變成 0.02 ns。


In [ ]:
# ============================================================
# 4. 使用者設定區：一般情況只修改這一區
# ============================================================

TOPOLOGY_PATH = Path(
    "/dicos_ui_home/chrysaliso/G4/TO_center_ion_1031/"
    "score606/step3_input.pdb"
)

DCD_PATH = Path(
    "/ceph/sharedfs/work/MYTLab/asher/center_ion_project/K/"
    "score_606/rep1/G4_TO_center_606.dcd"
)

SYSTEM_NAME = "K_606_rep1"
CURVE_COLOR = "royalblue"

# 核酸重原子同時用於 alignment 與 RMSD calculation。
ATOM_SELECTION = "nucleic and not element H"

# 1 代表使用每個 DCD frame。
TRAJECTORY_STRIDE = 1

# NAMD timestep = 2 fs，DCD 每 1000 steps 輸出一次。
TIME_PER_ORIGINAL_FRAME_NS = 0.002

# 圖片固定 Y 軸上限；設為 None 時依數據自動決定。
Y_MAX_ANGSTROM = 14.0

OUTPUT_DIR = Path("./results") / SYSTEM_NAME

print(f"System: {SYSTEM_NAME}")
print(f"Trajectory stride: {TRAJECTORY_STRIDE}")
print(
    "Sampled time interval: "
    f"{TIME_PER_ORIGINAL_FRAME_NS * TRAJECTORY_STRIDE:.3f} ns"
)
print(f"Output directory: {OUTPUT_DIR.resolve()}")


## 5. 載入與 selection 檢查

程式會確認 topology、DCD 存在，並顯示核酸重原子的數量。ATOM_SELECTION 必須同時用於 alignment 與 RMSD，避免前後使用不同原子集合。


In [ ]:
# ============================================================
# 6. 載入軌跡與建立 selection
# ============================================================

if not TOPOLOGY_PATH.is_file():
    raise FileNotFoundError(
        f"找不到 topology：{TOPOLOGY_PATH}\n"
        "請回到使用者設定區修改 TOPOLOGY_PATH。"
    )

if not DCD_PATH.is_file():
    raise FileNotFoundError(
        f"找不到 DCD：{DCD_PATH}\n"
        "請回到使用者設定區修改 DCD_PATH。"
    )

if not isinstance(TRAJECTORY_STRIDE, int) or TRAJECTORY_STRIDE <= 0:
    raise ValueError("TRAJECTORY_STRIDE 必須是正整數。")

if TIME_PER_ORIGINAL_FRAME_NS <= 0:
    raise ValueError("TIME_PER_ORIGINAL_FRAME_NS 必須大於 0。")

print(f"Loading trajectory: {DCD_PATH}")
traj = md.load(
    str(DCD_PATH),
    top=str(TOPOLOGY_PATH),
    stride=TRAJECTORY_STRIDE,
)

if traj.n_frames == 0:
    raise RuntimeError("軌跡沒有任何 frame。")

analysis_indices = traj.topology.select(ATOM_SELECTION)

if len(analysis_indices) == 0:
    raise ValueError(
        f"ATOM_SELECTION 沒有選到原子：{ATOM_SELECTION}\n"
        "請檢查 topology residue names 與 atom elements。"
    )

selected_atoms = [
    traj.topology.atom(int(atom_index))
    for atom_index in analysis_indices
]

selected_residue_indices = sorted({
    atom.residue.index
    for atom in selected_atoms
})

selected_residue_labels = [
    (
        f"{residue.name}{residue.resSeq}"
        if residue.resSeq is not None
        else f"{residue.name}{residue.index + 1}"
    )
    for residue in traj.topology.residues
    if residue.index in selected_residue_indices
]

print(f"Loaded frames: {traj.n_frames:,}")
print(f"Total atoms: {traj.n_atoms:,}")
print(f"Selected nucleic heavy atoms: {len(analysis_indices):,}")
print(f"Selected residues: {len(selected_residue_labels)}")
print("Residue labels:")
print(" ".join(selected_residue_labels))


## 7. 對齊、計算 RMSD 與保存數據

每個 frame 都以核酸重原子對齊至第 0 frame。完成對齊後，直接計算相同原子集合相對於第 0 frame 的 RMSD，不再呼叫會重新處理 alignment 的第二套函數。


In [ ]:
# ============================================================
# 8. Alignment、RMSD calculation 與數據輸出
# ============================================================

reference_frame = traj[0]

# 轉換矩陣由核酸重原子決定，並套用至完整軌跡。
traj.superpose(
    reference_frame,
    atom_indices=analysis_indices,
    ref_atom_indices=analysis_indices,
)

# MDTraj xyz 單位為 nm。
reference_xyz_nm = reference_frame.xyz[0, analysis_indices, :]
trajectory_xyz_nm = traj.xyz[:, analysis_indices, :]

coordinate_difference_nm = (
    trajectory_xyz_nm
    - reference_xyz_nm[np.newaxis, :, :]
)

# 每個 frame：先對每顆原子的 x/y/z 差平方加總，再對 atoms 取平均。
rmsd_nm = np.sqrt(
    np.mean(
        np.sum(coordinate_difference_nm ** 2, axis=2),
        axis=1,
    )
)
rmsd_angstrom = rmsd_nm * 10.0

sampled_frame = np.arange(traj.n_frames, dtype=int)
original_frame_index = sampled_frame * TRAJECTORY_STRIDE
time_ns = (
    original_frame_index
    * TIME_PER_ORIGINAL_FRAME_NS
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

npz_path = OUTPUT_DIR / "rmsd_nucleic_heavy_data.npz"
np.savez_compressed(
    npz_path,
    system_name=np.asarray(SYSTEM_NAME),
    curve_color=np.asarray(CURVE_COLOR),
    atom_selection=np.asarray(ATOM_SELECTION),
    analysis_indices=np.asarray(analysis_indices, dtype=int),
    sampled_frame=sampled_frame,
    original_frame_index=original_frame_index,
    time_ns=time_ns,
    rmsd_nm=rmsd_nm,
    rmsd_angstrom=rmsd_angstrom,
    trajectory_stride=np.asarray(TRAJECTORY_STRIDE),
    time_per_original_frame_ns=np.asarray(
        TIME_PER_ORIGINAL_FRAME_NS
    ),
)

csv_path = OUTPUT_DIR / "rmsd_nucleic_heavy_data.csv"
with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow([
        "sampled_frame",
        "original_frame_index",
        "time_ns",
        "rmsd_nm",
        "rmsd_angstrom",
    ])

    for values in zip(
        sampled_frame,
        original_frame_index,
        time_ns,
        rmsd_nm,
        rmsd_angstrom,
    ):
        writer.writerow([
            int(values[0]),
            int(values[1]),
            f"{values[2]:.6f}",
            f"{values[3]:.6f}",
            f"{values[4]:.6f}",
        ])

summary_path = OUTPUT_DIR / "run_summary.txt"
with summary_path.open("w", encoding="utf-8") as summary_file:
    summary_file.write(f"System: {SYSTEM_NAME}\n")
    summary_file.write(f"Topology: {TOPOLOGY_PATH}\n")
    summary_file.write(f"DCD: {DCD_PATH}\n")
    summary_file.write(f"Loaded frames: {traj.n_frames}\n")
    summary_file.write(f"Trajectory stride: {TRAJECTORY_STRIDE}\n")
    summary_file.write(
        "Time per original frame: "
        f"{TIME_PER_ORIGINAL_FRAME_NS} ns\n"
    )
    summary_file.write(f"Atom selection: {ATOM_SELECTION}\n")
    summary_file.write(
        f"Selected atoms: {len(analysis_indices)}\n"
    )
    summary_file.write("Alignment reference: frame 0\n")
    summary_file.write("RMSD reference: frame 0\n")
    summary_file.write("MDTraj coordinate unit: nm\n")

print(f"First-frame RMSD: {rmsd_angstrom[0]:.6f} Å")
print(f"Final sampled time: {time_ns[-1]:.3f} ns")
print(f"NPZ saved: {npz_path.resolve()}")
print(f"CSV saved: {csv_path.resolve()}")
print(f"Summary saved: {summary_path.resolve()}")


## 9. 從 NPZ 載入並繪圖

若只需調整圖形，可從此處往下執行，不必重新讀取 DCD。


In [ ]:
# ============================================================
# 10. 載入 NPZ
# ============================================================

npz_path = OUTPUT_DIR / "rmsd_nucleic_heavy_data.npz"

if not npz_path.is_file():
    raise FileNotFoundError(
        f"找不到 NPZ：{npz_path}\n"
        "請先執行 RMSD 計算 cell。"
    )

data = np.load(npz_path, allow_pickle=False)

required_keys = {
    "system_name",
    "curve_color",
    "time_ns",
    "rmsd_angstrom",
}

missing_keys = required_keys.difference(data.files)

if missing_keys:
    raise KeyError(f"NPZ 缺少欄位：{sorted(missing_keys)}")

plot_system_name = str(data["system_name"])
plot_color = str(data["curve_color"])
plot_time_ns = data["time_ns"]
plot_rmsd_angstrom = data["rmsd_angstrom"]

if len(plot_time_ns) != len(plot_rmsd_angstrom):
    raise ValueError("NPZ 中 time_ns 與 rmsd_angstrom 長度不同。")

print(f"Loaded: {npz_path.resolve()}")
print(f"Data points: {len(plot_time_ns):,}")


## 11. 繪製單一軌跡 RMSD

單一 DCD 只畫一條 RMSD 曲線，不畫 replicate mean、standard deviation 或 error band。X 軸範圍由實際軌跡時間決定。


In [ ]:
# ============================================================
# 12. RMSD 圖
# ============================================================

fig, ax = plt.subplots(figsize=(12, 8), dpi=300)

ax.plot(
    plot_time_ns,
    plot_rmsd_angstrom,
    color=plot_color,
    linewidth=2.5,
    linestyle="-",
    label=plot_system_name,
)

ax.set_xlabel("Time (ns)", fontsize=30)
ax.set_ylabel("RMSD (Å)", fontsize=30)
ax.set_xlim(
    float(plot_time_ns[0]),
    float(plot_time_ns[-1]),
)

if Y_MAX_ANGSTROM is None:
    automatic_y_max = max(
        1.0,
        float(np.max(plot_rmsd_angstrom)) * 1.10,
    )
    ax.set_ylim(0, automatic_y_max)
else:
    ax.set_ylim(0, Y_MAX_ANGSTROM)

ax.tick_params(
    axis="both",
    which="major",
    labelsize=24,
)

ax.legend(
    loc="upper left",
    fontsize=18,
    framealpha=0.9,
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.4,
)

fig.tight_layout()

png_path = OUTPUT_DIR / "figure_rmsd_nucleic_heavy.png"
pdf_path = OUTPUT_DIR / "figure_rmsd_nucleic_heavy.pdf"

fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    pdf_path,
    bbox_inches="tight",
)

plt.show()

print(f"PNG saved: {png_path.resolve()}")
print(f"PDF saved: {pdf_path.resolve()}")


## 13. 完成後確認

- ATOM_SELECTION 選到核酸重原子，不包含氫、配體、水或離子。
- 第 0 frame 的 RMSD 應接近 0 Å。
- STRIDE = 1 時，每個資料點間隔為 0.002 ns。
- 50,000 個原始 frames 約對應 100 ns。
- 圖形沒有 mean、SD、marker 或 error band。
- 若 RMSD 異常偏大，先檢查 G4 是否跨越週期邊界被拆開，再確認 topology 與 DCD 是否相符。

分析下一條 DCD 時，只需更換 DCD_PATH、SYSTEM_NAME、CURVE_COLOR，再由上往下重新執行。
